### **Data Exploration**

The project uses two Spotify datasets: one containing 2025 track data, and the other containing historical track data through 2025.

Through exploring the data, it revealed some inconsistencies to be cleaned before analysis:
* Track duration was stored in different units (*minutes and milliseconds*)
* Many numerical columns had inconsistent data types
* Album release dates was in 3 different formats: year, year-moonth, and full date
* Artist genres were stored differently across the datasets in stringified lists and comma separated values

In [1]:
import pandas as pd
import sys
sys.path.append("../scripts")
import utilities 
import sqlite3

In [2]:
spotify_2025 = pd.read_csv('../data/spotify_data_clean.csv')
spotify_prev = pd.read_csv('../data/track_data_final.csv')

In [3]:
spotify_2025.head()

,track_id,track_name,track_number,track_popularity,explicit,artist_name,artist_popularity,artist_followers,artist_genres,album_id,album_name,album_release_date,album_total_tracks,album_type,track_duration_min
0,3EJS5LyekDim1Tf5rBFmZl,Trippy Mane (ft. Project Pat),4,0,True,Diplo,77,2812821,moombahton,5QRFnGnBeMGePBKF2xTz5z,"d00mscrvll, Vol. 1",2025-10-31,9,album,1.55
1,1oQW6G2ZiwMuHqlPpP27DB,OMG!,1,0,True,Yelawolf,64,2363438,"country hip hop, southern hip hop",4SUmmwnv0xTjRcLdjczGg2,OMG!,2025-10-31,1,single,3.07
2,7mdkjzoIYlf1rx9EtBpGmU,Hard 2 Find,1,4,True,Riff Raff,48,193302,NaN,3E3zEAL8gUYWaLYB9L7gbp,Hard 2 Find,2025-10-31,1,single,2.55
3,67rW0Zl7oB3qEpD5YWWE5w,Still Get Like That (ft. Project Pat & Starrah),8,30,True,Diplo,77,2813710,moombahton,5QRFnGnBeMGePBKF2xTz5z,"d00mscrvll, Vol. 1",2025-10-31,9,album,1.69
4,15xptTfRBrjsppW0INUZjf,ride me like a harley,2,0,True,Rumelis,48,8682,dark r&b,06FDIpSHYmZAZoyuYtc7kd,come closer / ride me like a harley,2025-10-30,2,single,2.39


In [4]:
spotify_prev.head()

,track_id,track_name,track_number,track_popularity,track_duration_ms,explicit,artist_name,artist_popularity,artist_followers,artist_genres,album_id,album_name,album_release_date,album_total_tracks,album_type
0,6pymOcrCnMuCWdgGVTvUgP,3,57,61,213173,False,Britney Spears,80.0,17755451.0,['pop'],325wcm5wMnlfjmKZ8PXIIn,The Singles Collection,2009-11-09,58,compilation
1,2lWc1iJlz2NVcStV5fbtPG,Clouds,1,67,158760,False,BUNT.,69.0,293734.0,['stutter house'],2ArRQNLxf9t0O0gvmG5Vsj,Clouds,2023-01-13,1,single
2,1msEuwSBneBKpVCZQcFTsU,Forever & Always (Taylor’s Version),11,63,225328,False,Taylor Swift,100.0,145396321.0,[],4hDok0OAJd57SGIT8xuWJH,Fearless (Taylor's Version),2021-04-09,26,album
3,7bcy34fBT2ap1L4bfPsl9q,I Didn't Change My Number,2,72,158463,True,Billie Eilish,90.0,118692183.0,[],0JGOiO34nwfUdDrD612dOp,Happier Than Ever,2021-07-30,16,album
4,0GLfodYacy3BJE7AI3A8en,Man Down,7,57,267013,False,Rihanna,90.0,68997177.0,[],5QG3tjE5L9F6O2vCAPph38,Loud,2010-01-01,13,album


In [5]:
spotify_2025.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8582 entries, 0 to 8581
Data columns (total 15 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   track_id            8582 non-null   object 
 1   track_name          8582 non-null   object 
 2   track_number        8582 non-null   int64  
 3   track_popularity    8582 non-null   int64  
 4   explicit            8582 non-null   bool   
 5   artist_name         8579 non-null   object 
 6   artist_popularity   8582 non-null   int64  
 7   artist_followers    8582 non-null   int64  
 8   artist_genres       5221 non-null   object 
 9   album_id            8582 non-null   object 
 10  album_name          8582 non-null   object 
 11  album_release_date  8582 non-null   object 
 12  album_total_tracks  8582 non-null   int64  
 13  album_type          8582 non-null   object 
 14  track_duration_min  8582 non-null   float64
dtypes: bool(1), float64(1), int64(5), object(8)
memory usag

In [6]:
spotify_prev.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8778 entries, 0 to 8777
Data columns (total 15 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   track_id            8778 non-null   object 
 1   track_name          8776 non-null   object 
 2   track_number        8778 non-null   int64  
 3   track_popularity    8778 non-null   int64  
 4   track_duration_ms   8778 non-null   int64  
 5   explicit            8778 non-null   bool   
 6   artist_name         8774 non-null   object 
 7   artist_popularity   8774 non-null   float64
 8   artist_followers    8774 non-null   float64
 9   artist_genres       8774 non-null   object 
 10  album_id            8778 non-null   object 
 11  album_name          8776 non-null   object 
 12  album_release_date  8778 non-null   object 
 13  album_total_tracks  8778 non-null   int64  
 14  album_type          8778 non-null   object 
dtypes: bool(1), float64(2), int64(4), object(8)
memory usag

In [7]:
# checking unique values for album_release_date for cleaning
print(spotify_prev['album_release_date'].str.len().value_counts())

album_release_date
10    8577
4      198
7        3
Name: count, dtype: int64


In [8]:
# creating filters to locate values with lengths 4 & 7
format4 = spotify_prev['album_release_date'].astype(str).str.strip().str.len() == 4
format7 = spotify_prev['album_release_date'].astype(str).str.strip().str.len() == 7

In [9]:
spotify_prev.loc[format4, 'album_release_date']

27      2009
49      1980
67      2007
87      2007
92      2005
        ... 
8638    2000
8675    2007
8713    1999
8729    1995
8760    2009
Name: album_release_date, Length: 198, dtype: object

In [10]:
spotify_prev.loc[format7, 'album_release_date']

2394    1981-12
4752    1967-09
4887    1971-08
Name: album_release_date, dtype: object

In [ ]:
# checking for overlapping tracks between the two datasets
merged = spotify_prev.merge(spotify_2025, how = 'inner', on = 'track_id')
merged.head()

,track_id,track_name_x,track_number_x,track_popularity_x,track_duration_ms,explicit_x,artist_name_x,artist_popularity_x,artist_followers_x,artist_genres_x,...,artist_name_y,artist_popularity_y,artist_followers_y,artist_genres_y,album_id_y,album_name_y,album_release_date_y,album_total_tracks_y,album_type_y,track_duration_min
0,6pymOcrCnMuCWdgGVTvUgP,3,57,61,213173,False,Britney Spears,80.0,17755451.0,['pop'],...,Britney Spears,80,17755451,pop,325wcm5wMnlfjmKZ8PXIIn,The Singles Collection,2009-11-09,58,compilation,3.55
1,2lWc1iJlz2NVcStV5fbtPG,Clouds,1,67,158760,False,BUNT.,69.0,293734.0,['stutter house'],...,BUNT.,69,293734,stutter house,2ArRQNLxf9t0O0gvmG5Vsj,Clouds,2023-01-13,1,single,2.64
2,1msEuwSBneBKpVCZQcFTsU,Forever & Always (Taylor’s Version),11,63,225328,False,Taylor Swift,100.0,145396321.0,[],...,Taylor Swift,100,145396321,"country, pop, indie, folk",4hDok0OAJd57SGIT8xuWJH,Fearless (Taylor's Version),2021-04-09,26,album,3.75
3,7bcy34fBT2ap1L4bfPsl9q,I Didn't Change My Number,2,72,158463,True,Billie Eilish,90.0,118692183.0,[],...,Billie Eilish,90,118692183,"alternative pop, electropop, dark pop",0JGOiO34nwfUdDrD612dOp,Happier Than Ever,2021-07-30,16,album,2.64
4,0GLfodYacy3BJE7AI3A8en,Man Down,7,57,267013,False,Rihanna,90.0,68997177.0,[],...,Rihanna,90,68997177,"pop, r&b",5QG3tjE5L9F6O2vCAPph38,Loud,2010-01-01,13,album,4.45


> 'spotify_prev' contains historical through 2025 tracks, 'spotify_2025' contain 2025 specific tracks

In [ ]:
# checking if cleaning for 'artist_genres' is needed
spotify_prev["artist_genres"].apply(type).value_counts()

artist_genres
<class 'str'>      8774
<class 'float'>       4
Name: count, dtype: int64

In [13]:
spotify_2025["artist_genres"].apply(type).value_counts()

artist_genres
<class 'str'>      5221
<class 'float'>    3361
Name: count, dtype: int64

In [ ]:
spotify_prev["artist_genres"].head(10).tolist() 
# not actual lists, they're stringified lists

["['pop']",
 "['stutter house']",
 '[]',
 '[]',
 '[]',
 "['dream pop']",
 '[]',
 "['brazilian bass', 'electronic', 'slap house', 'electro house']",
 "['soft pop']",
 '[]']

In [ ]:
spotify_2025["artist_genres"].head(10).tolist() 
# not actual lists, they're comma separated strings

['moombahton',
 'country hip hop, southern hip hop',
 nan,
 'moombahton',
 'dark r&b',
 'dark r&b',
 nan,
 'dark r&b',
 nan,
 'nigerian drill, alté, afro adura, afrobeats, afrobeat, afroswing']

> 'artist_genres' column is stored differently between the two datasets. Historical data contained stringfied lists, while the 2025 dataset uses comma separated string. These formats need to be standardized before genres are separated and analyzed individually.